# NeuroDetect — Brain Tumor Detection
**Course**: Computer Vision | Christian Mata, PhD

**Dataset**: LGG Brain MRI Segmentation (Kaggle — mateuszbuda)

**Task**: Train / Test Split

**Group members**: Martin Çaro, Edison Zyberaj

# Train / Test Split
**Requires 00_setup.** Loads all images, derives labels, splits, saves to Drive.

## 1. Mount Drive and check setup

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/CV_Project'
os.makedirs(DRIVE + '/brain/images', exist_ok=True)
os.makedirs(DRIVE + '/brain/masks',  exist_ok=True)
os.makedirs(DRIVE + '/splits',       exist_ok=True)
os.makedirs(DRIVE + '/models',       exist_ok=True)
os.makedirs(DRIVE + '/results',      exist_ok=True)
print('Drive mounted ✓  —  ' + DRIVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted ✓  —  /content/drive/MyDrive/CV_Project


In [5]:
import os
assert os.path.isdir(DRIVE + '/brain/images'),     '❌ Run 00_setup.ipynb first.'
import glob
n = len(glob.glob(DRIVE + '/brain/images/*.png'))
print(f'Found {n} PNG images in Drive ✓')

Found 3929 PNG images in Drive ✓


## 2. Load images and derive labels

In [7]:
import glob
import numpy as np
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
import threading
import time

image_pngs = sorted(glob.glob(DRIVE + '/brain/images/*.png'))
mask_pngs  = sorted(glob.glob(DRIVE + '/brain/masks/*.png'))

# Thread-safe counter
lock    = threading.Lock()
counter = {'images': 0, 'labels': 0}
total   = len(image_pngs)

def load_image(path):
    result = np.array(Image.open(path), dtype=np.float32) / 255.0
    with lock:
        counter['images'] += 1
        done = counter['images']
        if done % 100 == 0 or done == total:
            print(f'  [images] {done}/{total} — {path.split("/")[-1]}')
    return result

def load_label(path):
    result = 1 if np.array(Image.open(path).convert('L')).max() > 0 else 0
    with lock:
        counter['labels'] += 1
        done = counter['labels']
        if done % 100 == 0 or done == total:
            print(f'  [labels] {done}/{total} — {path.split("/")[-1]}')
    return result

print(f'Loading {total} images with 8 threads...')
t0 = time.time()

with ThreadPoolExecutor(max_workers=8) as ex:
    x_data = np.array(list(ex.map(load_image, image_pngs)))
    labels = np.array(list(ex.map(load_label, mask_pngs)))

print(f'\nDone in {time.time() - t0:.1f}s')
print(f'x_data shape : {x_data.shape}')
print(f'Tumor slices : {labels.sum()} ({labels.mean()*100:.1f}%)')
print(f'No-tumor     : {(1-labels).sum()} ({(1-labels).mean()*100:.1f}%)')

Loading 3929 images with 8 threads...
  [images] 100/3929 — TCGA_CS_5393_19990606_7.png
  [images] 200/3929 — TCGA_CS_6188_20010812_14.png
  [images] 300/3929 — TCGA_CS_6667_20011105_19.png
  [images] 400/3929 — TCGA_DU_5851_19950428_13.png
  [images] 500/3929 — TCGA_DU_5853_19950823_34.png
  [images] 600/3929 — TCGA_DU_5871_19941206_36.png
  [images] 700/3929 — TCGA_DU_5874_19950510_3.png
  [images] 800/3929 — TCGA_DU_6400_19830518_38.png
  [images] 900/3929 — TCGA_DU_6404_19850629_3.png
  [images] 1000/3929 — TCGA_DU_6407_19860514_18.png
  [images] 1100/3929 — TCGA_DU_6408_19860521_6.png
  [images] 1200/3929 — TCGA_DU_7010_19860307_52.png
  [images] 1300/3929 — TCGA_DU_7014_19860618_4.png
  [images] 1400/3929 — TCGA_DU_7294_19890104_12.png
  [images] 1500/3929 — TCGA_DU_7299_19910417_9.png
  [images] 1600/3929 — TCGA_DU_7302_19911203_4.png
  [images] 1700/3929 — TCGA_DU_7309_19960831_24.png
  [images] 1800/3929 — TCGA_DU_8163_19961119_9.png
  [images] 1900/3929 — TCGA_DU_8166_1997032

## 3. Train / test split

In [8]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_data, labels, test_size=0.2, random_state=42
)
# Also split paths (needed by segmentation notebook)
idx_all = list(range(len(image_pngs)))
idx_train, idx_test = train_test_split(idx_all, test_size=0.2, random_state=42)

train_img = [image_pngs[i] for i in idx_train]
train_msk = [mask_pngs[i]  for i in idx_train]
val_img   = [image_pngs[i] for i in idx_test]
val_msk   = [mask_pngs[i]  for i in idx_test]

print(f'Train: {len(x_train)} images')
print(f'Test:  {len(x_test)}  images')

Train: 3143 images
Test:  786  images


## 4. Save splits to Drive

In [9]:
import json

np.save(DRIVE + '/splits/x_train.npy', x_train)
np.save(DRIVE + '/splits/x_test.npy',  x_test)
np.save(DRIVE + '/splits/y_train.npy', y_train)
np.save(DRIVE + '/splits/y_test.npy',  y_test)

seg_paths = {'train_img': train_img, 'train_msk': train_msk,
             'val_img': val_img,     'val_msk':   val_msk}
with open(DRIVE + '/splits/seg_paths.json', 'w') as f:
    json.dump(seg_paths, f)

print('Saved to Drive ✓')
print('  splits/x_train.npy,  x_test.npy')
print('  splits/y_train.npy,  y_test.npy')
print('  splits/seg_paths.json')
print()
print('Next steps: run 02_classification.ipynb and 03_segmentation.ipynb')

Saved to Drive ✓
  splits/x_train.npy,  x_test.npy
  splits/y_train.npy,  y_test.npy
  splits/seg_paths.json

Next steps: run 02_classification.ipynb and 03_segmentation.ipynb
